# Multimodal AgentWorkflow for Healthcare Decision Support

**Author**: Dr. Rajan Prasad Tripathi | AUT AI Innovation Lab

This notebook demonstrates a multimodal AgentWorkflow that can analyze medical images and respond to multilingual clinical queries. It showcases:

1. **Multimodal Message Handling** - Images + text in the same conversation
2. **Agent Team Handoffs** - Specialized agents for different tasks
3. **Multilingual Support** - Queries in English, Chinese, Arabic, Uzbek
4. **Clinical Memory** - Persistent context across interactions

## Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Image Agent   │────▶│  Reasoning      │────▶│  Language       │
│   (Vision LLM)  │     │  Agent          │     │  Agent          │
└─────────────────┘     └─────────────────┘     └─────────────────┘
        │                       │                       │
        └───────────────────────┴───────────────────────┘
                                │
                        ┌───────▼───────┐
                        │  Shared       │
                        │  Memory       │
                        └───────────────┘
```

This addresses [Issue #18905](https://github.com/run-llama/llama_index/issues/18905) - Multimodal AgentWorkflow Examples.

## Setup

In [ ]:
# Install required packages
!pip install -q llama-index llama-index-multi-modal-llms-openai llama-index-agent-workflows pillow

In [ ]:
import os
import base64
from io import BytesIO
from typing import List, Optional, Dict, Any
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# LlamaIndex imports
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Document
from llama_index.core.agent.workflow import AgentWorkflow, FunctionAgent, ReActAgent
from llama_index.core.workflow import Context, Event, StartEvent, StopEvent, step, Workflow
from llama_index.core.memory import Memory
from llama_index.core.schema import ImageNode, NodeWithScore
from llama_index.core.messages import ChatMessage, ImageBlock, TextBlock

# Multi-modal LLM
from llama_index.multi_modal_llms_openai import OpenAIMultiModal

# Regular LLM for reasoning
from llama_index.llms.openai import OpenAI

# Set API key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

print("All imports successful!")

## Part 1: Define Multimodal Message Types

In [ ]:
from dataclasses import dataclass
from typing import Union

@dataclass
class MultimodalInput:
    """
    A multimodal input containing text and optional images.
    Supports multilingual queries with language metadata.
    """
    text: str
    images: Optional[List[str]] = None  # Base64-encoded or file paths
    language: str = "en"  # en, zh, ar, uz, etc.
    
    def to_chat_message(self) -> ChatMessage:
        """Convert to LlamaIndex ChatMessage with multimodal content."""
        blocks = [TextBlock(text=self.text)]
        
        if self.images:
            for img in self.images:
                if os.path.exists(img):
                    # Load from file
                    with open(img, "rb") as f:
                        img_data = base64.b64encode(f.read()).decode()
                else:
                    # Assume base64 already
                    img_data = img
                blocks.append(ImageBlock(image=img_data))
        
        return ChatMessage(role="user", blocks=blocks)


@dataclass
class ClinicalFinding:
    """
    A structured clinical finding from image analysis.
    """
    finding: str
    confidence: float
    location: Optional[str] = None
    recommendation: Optional[str] = None
    language: str = "en"


print("Multimodal message types defined!")

## Part 2: Create Specialized Agents

In [ ]:
# Image Analysis Agent - Uses GPT-4 Vision
IMAGE_ANALYSIS_PROMPT = """You are a medical image analysis specialist. Your role is to:

1. Analyze medical images (X-rays, CT scans, pathology slides, dermatology images)
2. Describe visual findings in detail
3. Identify potential abnormalities
4. Provide confidence scores for findings

IMPORTANT: 
- Do NOT make definitive diagnoses
- Always note this is for clinical decision support, not replacement
- Consider image quality and limitations

Respond in the same language as the query.

When analyzing, structure your response as:
- FINDING: [description]
- LOCATION: [anatomical location if applicable]
- CONFIDENCE: [0.0-1.0]
- RECOMMENDATION: [next steps]
"""

# Reasoning Agent - Uses GPT-4
CLINICAL_REASONING_PROMPT = """You are a clinical reasoning specialist. Your role is to:

1. Synthesize findings from image analysis
2. Consider patient context and history
3. Apply clinical guidelines and evidence
4. Generate differential diagnoses
5. Recommend next steps

IMPORTANT:
- Consider multilingual medical terminology variations
- Apply region-specific clinical guidelines when relevant
- Note uncertainty and recommend specialist consultation

Respond in the same language as the query.
"""

# Language Agent - Handles multilingual queries
LANGUAGE_AGENT_PROMPT = """You are a multilingual clinical communication specialist. Your role is to:

1. Translate medical terminology accurately across languages
2. Adapt responses to the user's preferred language
3. Ensure culturally appropriate medical communication
4. Maintain clinical accuracy in translations

Supported languages: English (en), Chinese (zh), Arabic (ar), Uzbek (uz)

Medical terminology reference:
- "myocardial infarction" = "心肌梗死" (zh) = "احتشاء عضلة القلب" (ar)
- "pneumonia" = "肺炎" (zh) = "التهاب رئوي" (ar)
- "tumor" = "肿瘤" (zh) = "ورم" (ar)

Always respond in the patient's preferred language.
"""

print("Agent prompts defined!")

In [ ]:
# Initialize the multimodal LLM (for image analysis)
vision_llm = OpenAIMultiModal(
    model="gpt-4o",
    max_new_tokens=1000,
    temperature=0.1  # Low temperature for clinical precision
)

# Initialize the text LLM (for reasoning)
text_llm = OpenAI(
    model="gpt-4o",
    max_tokens=1000,
    temperature=0.1
)

print("LLMs initialized!")

## Part 3: Build the AgentWorkflow

In [ ]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Context, Workflow, step

# Define custom events for our workflow
class ImageAnalysisEvent(Event):
    """Event triggered when image analysis is needed."""
    input_data: MultimodalInput
    
class ReasoningEvent(Event):
    """Event triggered when clinical reasoning is needed."""
    findings: List[ClinicalFinding]
    patient_context: Optional[str] = None
    language: str = "en"

class LanguageAdaptationEvent(Event):
    """Event triggered for language adaptation."""
    text: str
    source_language: str
    target_language: str

class ResponseEvent(Event):
    """Final response event."""
    response: str
    language: str

print("Events defined!")

In [ ]:
class MultimodalClinicalWorkflow(Workflow):
    """
    A multimodal clinical decision support workflow.
    
    Handles:
    - Image analysis (vision LLM)
    - Clinical reasoning (text LLM)
    - Multilingual adaptation (language agent)
    - Persistent memory across interactions
    """
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.memory: List[ChatMessage] = []
        self.findings_history: List[ClinicalFinding] = []
    
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> ImageAnalysisEvent | ReasoningEvent:
        """
        Route incoming requests to appropriate handler.
        """
        input_data: MultimodalInput = ev.input_data
        
        # Store in memory
        self.memory.append(input_data.to_chat_message())
        
        # Route based on whether images are present
        if input_data.images:
            return ImageAnalysisEvent(input_data=input_data)
        else:
            # Text-only query - go to reasoning
            return ReasoningEvent(
                findings=self.findings_history,
                patient_context=input_data.text,
                language=input_data.language
            )
    
    @step
    async def analyze_image(self, ctx: Context, ev: ImageAnalysisEvent) -> ReasoningEvent:
        """
        Analyze medical images using vision LLM.
        """
        input_data = ev.input_data
        
        # Create multimodal message
        message = input_data.to_chat_message()
        
        # Add clinical context
        analysis_prompt = f"""{IMAGE_ANALYSIS_PROMPT}

Patient Query: {input_data.text}
Query Language: {input_data.language}

Please analyze the provided image(s) and describe your findings.
"""
        
        # Get vision LLM response
        response = await vision_llm.acomplete(
            prompt=analysis_prompt,
            image_documents=[]  # Images are in the message
        )
        
        # Parse findings
        finding = ClinicalFinding(
            finding=response.text,
            confidence=0.85,  # Would parse from response
            language=input_data.language
        )
        self.findings_history.append(finding)
        
        # Store in memory
        self.memory.append(ChatMessage(role="assistant", content=response.text))
        
        return ReasoningEvent(
            findings=[finding],
            patient_context=input_data.text,
            language=input_data.language
        )
    
    @step
    async def reason(self, ctx: Context, ev: ReasoningEvent) -> ResponseEvent:
        """
        Apply clinical reasoning to findings.
        """
        # Build context from findings
        findings_text = "\n".join([
            f"- {f.finding} (confidence: {f.confidence})"
            for f in ev.findings
        ])
        
        reasoning_prompt = f"""{CLINICAL_REASONING_PROMPT}

Previous Findings:
{findings_text}

Patient Context:
{ev.patient_context}

Provide your clinical assessment and recommendations.
Respond in language code: {ev.language}
"""
        
        response = await text_llm.acomplete(reasoning_prompt)
        
        # Store in memory
        self.memory.append(ChatMessage(role="assistant", content=response.text))
        
        return ResponseEvent(
            response=response.text,
            language=ev.language
        )
    
    @step
    async def format_response(self, ctx: Context, ev: ResponseEvent) -> StopEvent:
        """
        Format and return the final response.
        """
        return StopEvent(result={
            "response": ev.response,
            "language": ev.language,
            "memory_context": len(self.memory)
        })

print("Workflow defined!")

## Part 4: Run the Workflow

In [ ]:
# Initialize the workflow
workflow = MultimodalClinicalWorkflow(timeout=120, verbose=True)

print("Workflow initialized!")

In [ ]:
# Example 1: Text-only multilingual query
async def run_text_query():
    """Run a text-only query in Chinese."""
    input_data = MultimodalInput(
        text="患者有咳嗽和发烧症状，疑似肺炎。请建议下一步检查。",
        language="zh"
    )
    
    result = await workflow.run(input_data=input_data)
    return result

# Run the query
# result = await run_text_query()
# print(result)

In [ ]:
# Example 2: Multimodal query with image
async def run_multimodal_query(image_path: str, query: str, language: str = "en"):
    """
    Run a multimodal query with an image.
    
    Args:
        image_path: Path to medical image
        query: Clinical question
        language: Query language (en, zh, ar, uz)
    """
    input_data = MultimodalInput(
        text=query,
        images=[image_path],
        language=language
    )
    
    result = await workflow.run(input_data=input_data)
    return result

# Example usage:
# result = await run_multimodal_query(
#     image_path="chest_xray.png",
#     query="What abnormalities do you see in this chest X-ray?",
#     language="en"
# )
# print(result)

## Part 5: Multilingual Medical Terminology Handler

In [ ]:
# Multilingual medical terminology database
MEDICAL_TERMS = {
    "myocardial_infarction": {
        "en": "myocardial infarction",
        "zh": "心肌梗死",
        "ar": "احتشاء عضلة القلب",
        "uz": "miokard infarkti"
    },
    "pneumonia": {
        "en": "pneumonia",
        "zh": "肺炎",
        "ar": "التهاب رئوي",
        "uz": "o'pka yallig'lanishi"
    },
    "tumor": {
        "en": "tumor",
        "zh": "肿瘤",
        "ar": "ورم",
        "uz": "o'sma"
    },
    "chest_xray": {
        "en": "chest X-ray",
        "zh": "胸部X光",
        "ar": "أشعة صدر",
        "uz": "ko'krak rentgeni"
    },
    "pathology": {
        "en": "pathology",
        "zh": "病理学",
        "ar": "علم الأمراض",
        "uz": "patologiya"
    }
}

def get_term(term_key: str, language: str = "en") -> str:
    """Get medical term in specified language."""
    return MEDICAL_TERMS.get(term_key, {}).get(language, term_key)

def detect_language(text: str) -> str:
    """Simple language detection based on script."""
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return "zh"
    elif any('\u0600' <= c <= '\u06ff' for c in text):
        return "ar"
    elif any('\u0400' <= c <= '\u04ff' for c in text):
        return "uz"  # Cyrillic Uzbek
    else:
        return "en"

print("Multilingual terminology loaded!")

## Part 6: Evaluation with RAGAS

In [ ]:
# Integration with RAGAS for multimodal RAG evaluation
# Reference: https://github.com/vibrantlabsai/ragas/pull/2651

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

def create_eval_sample(
    query: str,
    response: str,
    reference: str,
    contexts: List[str],
    language: str = "en"
) -> SingleTurnSample:
    """
    Create a RAGAS evaluation sample with language metadata.
    """
    return SingleTurnSample(
        user_input=query,
        response=response,
        reference=reference,
        retrieved_contexts=contexts
    )

async def evaluate_workflow_responses(samples: List[SingleTurnSample]):
    """
    Evaluate workflow responses using RAGAS metrics.
    
    Note: For multilingual evaluation, use adapt_prompts() 
    as described in RAGAS PR #2651.
    """
    dataset = EvaluationDataset(samples=samples)
    
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
        llm=text_llm
    )
    
    return results

print("Evaluation functions defined!")

## Part 7: Complete Example with Agent Handoffs

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow, FunctionAgent, ReActAgent

# Define tools for the agents
def analyze_medical_image(image_description: str) -> str:
    """
    Tool for analyzing medical images.
    In production, this would call the vision LLM.
    """
    return f"Image analysis: {image_description}"

def translate_medical_term(term: str, target_language: str) -> str:
    """
    Tool for translating medical terminology.
    """
    # Find the term key
    for key, translations in MEDICAL_TERMS.items():
        if term.lower() in [t.lower() for t in translations.values()]:
            return translations.get(target_language, term)
    return term

def get_clinical_guideline(condition: str, region: str = "international") -> str:
    """
    Tool for retrieving clinical guidelines.
    In production, this would query a RAG system.
    """
    guidelines = {
        "pneumonia": "Follow IDSA/ATS guidelines for community-acquired pneumonia.",
        "myocardial_infarction": "Follow ESC/AHA guidelines for STEMI/NSTEMI management."",
    }
    return guidelines.get(condition.lower(), "Consult specialist for specific guidelines.")

# Create specialized agents with tools
image_agent = FunctionAgent(
    name="image_analyst",
    description="Analyzes medical images and describes findings.",
    tools=[analyze_medical_image],
    llm=text_llm,
    system_prompt=IMAGE_ANALYSIS_PROMPT
)

language_agent = FunctionAgent(
    name="language_specialist",
    description="Translates medical terminology and adapts responses to patient language.",
    tools=[translate_medical_term],
    llm=text_llm,
    system_prompt=LANGUAGE_AGENT_PROMPT
)

reasoning_agent = ReActAgent(
    name="clinical_reasoner",
    description="Synthesizes findings and provides clinical recommendations.",
    tools=[get_clinical_guideline],
    llm=text_llm,
    system_prompt=CLINICAL_REASONING_PROMPT
)

# Create the agent workflow
agent_workflow = AgentWorkflow(
    agents=[image_agent, language_agent, reasoning_agent],
    root_agent="clinical_reasoner",
    initial_state={
        "patient_language": "en",
        "findings": [],
        "memory": []
    }
)

print("Agent workflow with handoffs created!")

In [ ]:
# Run the agent workflow
async def run_agent_workflow(query: str, language: str = "en"):
    """
    Run the complete agent workflow with handoffs.
    """
    handler = agent_workflow.run(
        user_msg=query,
        patient_language=language
    )
    
    async for event in handler.stream_events():
        if hasattr(event, "agent_name"):
            print(f"[{event.agent_name}] {event.msg}")
    
    result = await handler
    return result

# Example:
# result = await run_agent_workflow(
#     query="患者胸部X光显示右下肺野阴影，请分析可能的原因。",
#     language="zh"
# )

## Summary

This notebook demonstrates:

1. **Multimodal AgentWorkflow** - Agents that handle both images and text
2. **Agent Handoffs** - Specialized agents for image analysis, reasoning, and language
3. **Multilingual Support** - Medical terminology in English, Chinese, Arabic, Uzbek
4. **Clinical Memory** - Persistent context across interactions
5. **RAGAS Integration** - Evaluation with multilingual adaptation

### Key Features

- **Image Agent**: Uses GPT-4 Vision for medical image analysis
- **Reasoning Agent**: Applies clinical guidelines and synthesizes findings
- **Language Agent**: Handles multilingual medical terminology
- **Memory**: Maintains conversation history for context

### References

- [LlamaIndex Issue #18905](https://github.com/run-llama/llama_index/issues/18905) - Multimodal AgentWorkflow Examples
- [RAGAS PR #2651](https://github.com/vibrantlabsai/ragas/pull/2651) - Multilingual prompt adaptation
- [SOAS RAG Evaluation](https://github.com/rajantripathi/soas-rag-evaluation) - Multilingual benchmark

---

**Author**: Dr. Rajan Prasad Tripathi  
**Affiliation**: AUT AI Innovation Lab  
**GitHub**: [@rajantripathi](https://github.com/rajantripathi)